# The Intender Gap: Exploratory Data Analysis

## Business Question

**Who are the developers who intend to use AI tools but haven't yet — and how do they compare to those who use AI tools daily and those who have rejected AI entirely?**

Stack Overflow's 2025 Developer Survey shows that 84% of developers now use or plan to use AI tools. But 5.3% of all respondents sit in an unusual middle ground: they have declared intent to adopt but haven't acted. A PM at any AI developer tools company (GitHub Copilot, Cursor, Claude Code) needs to know:

- **Who** these intenders are — their role, experience, and context
- **What** distinguishes them from developers who actively use AI every day
- **What** separates them from developers who have flatly rejected AI tools

The answers tell us where to intervene and what messaging will move them over the line.

---

**Dataset:** Stack Overflow 2025 Annual Developer Survey (n = 49,191 raw; 33,720 valid after target variable filtering)  
**Target column:** `AISelect` — "Do you currently use AI tools in your development process?"  
**Primary analysis groups:** Intenders vs. Rejectors (both are non-users — this isolates psychological and attitudinal drivers of intent, holding actual access and habit constant)


## Setup: Libraries and Global Styling

We import the core data and visualization libraries here. We also define a consistent color scheme that will be used across every chart in this project — this makes it easier for a reader to track groups across multiple visualizations without re-reading the legend each time.

Color assignments follow a deliberate logic:
- **Teal** for active users — they're already engaged, a 'go' signal
- **Amber** for casual users — warm but not fully committed
- **Purple** for intenders — the primary group of interest
- **Coral** for rejectors — a contrast signal, the group we're differentiating from


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Consistent color scheme across all notebooks ──────────────────────────────
GROUP_COLORS = {
    'active':    '#1D9E75',   # teal  — active users (daily + weekly)
    'casual':    '#BA7517',   # amber — casual users (monthly + infrequently)
    'intender':  '#534AB7',   # purple — intenders (no, but plan to)
    'rejector':  '#993C1D',   # coral  — rejectors (no, and don't plan to)
}

# Ordered list for consistent display in charts
GROUP_ORDER     = ['active', 'casual', 'intender', 'rejector']
GROUP_LABELS    = {
    'active':   'Active Users',
    'casual':   'Casual Users',
    'intender': 'Intenders',
    'rejector': 'Rejectors',
}

# ── Output paths ──────────────────────────────────────────────────────────────
FIGURES_DIR = Path('../outputs/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Global matplotlib style ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

print('Libraries loaded. Output directory:', FIGURES_DIR.resolve())


## Load and Inspect the Raw Data

We load the raw CSV from `data/raw/` and immediately confirm a few structural facts: total row count, the presence of the key target column `AISelect`, and a sample of the raw values. We use `low_memory=False` because some columns have mixed types (strings mixed with numbers), which causes pandas to warn otherwise.

Importantly, we **never modify** the raw file. All filtering and transformation happens here in code.


In [ ]:
RAW_DATA_PATH = Path('../data/raw/survey_results_public.csv')

df_raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)

print(f'Raw dataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print()
print('AISelect raw value counts (including nulls):')
print(df_raw['AISelect'].value_counts(dropna=False).to_string())


## Step 1: Filter to Valid Responses and Assign Group Labels

15,471 respondents left `AISelect` blank. We confirmed via a dropout-curve analysis (see DECISIONS.md, D-07) that these are **Missing Completely at Random (MCAR)** — the missingness pattern does not correlate with any demographic variable. Because the missingness is random, we drop these rows rather than imputing, since imputation would introduce bias without a systematic signal to anchor on.

After dropping, we assign each respondent to one of four groups based on their `AISelect` response:
- **Active users:** Use AI daily or weekly (the fully adopted group)
- **Casual users:** Use AI monthly or infrequently (partially adopted)
- **Intenders:** No current use, but plan to adopt soon ← *our primary target*
- **Rejectors:** No current use and no intent to adopt ← *our comparison group*


In [ ]:
# Drop rows where AISelect is null — these are MCAR (see DECISIONS.md D-07)
df_valid = df_raw[df_raw['AISelect'].notna()].copy()

print(f'Rows after dropping AISelect nulls: {len(df_valid):,}')
print(f'Rows dropped: {len(df_raw) - len(df_valid):,}')
print()

# ── Map AISelect responses to four group labels ────────────────────────────────
aiselect_to_group = {
    'Yes, I use AI tools daily':                   'active',
    'Yes, I use AI tools weekly':                  'active',
    'Yes, I use AI tools monthly or infrequently': 'casual',
    'No, but I plan to soon':                      'intender',
    'No, and I don\'t plan to':                    'rejector',
}

df_valid['group'] = df_valid['AISelect'].map(aiselect_to_group)

print('Group assignment counts:')
group_counts = df_valid['group'].value_counts().reindex(GROUP_ORDER)
group_pct    = (group_counts / len(df_valid) * 100).round(1)
group_summary = pd.DataFrame({'Count': group_counts, 'Percent (%)': group_pct})
print(group_summary.to_string())


## Chart 1: How Developers Are Distributed Across AI Adoption Groups

Before any comparative analysis, we need to see the raw shape of the population. This chart shows how many of the 33,720 valid respondents fall into each group. The intender group is small in absolute terms — but the question is what distinguishes them, not how many there are.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

bar_labels  = [GROUP_LABELS[g] for g in GROUP_ORDER]
bar_colors  = [GROUP_COLORS[g] for g in GROUP_ORDER]
bar_heights = [group_counts[g] for g in GROUP_ORDER]

bars = ax.bar(bar_labels, bar_heights, color=bar_colors, edgecolor='white', linewidth=0.8)

# Annotate each bar with count and percentage
for bar, count, g in zip(bars, bar_heights, GROUP_ORDER):
    pct = group_pct[g]
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 200,
        f'{count:,}\n({pct}%)',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.set_title('AI Tool Adoption: 64.7% of Developers Use AI Tools, 5.3% Intend To', pad=12)
ax.set_ylabel('Number of Respondents')
ax.set_xlabel('')
ax.set_ylim(0, max(bar_heights) * 1.18)

# One-sentence takeaway
ax.annotate(
    'The intender group (5.3%) is small but represents a distinct, high-intent segment with declared motivation but no action — the most tractable growth opportunity.',
    xy=(0.5, -0.18), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_01_aiselect_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_01_aiselect_distribution.png')


## Step 2: Four-Group Comparison — AI Attitude Dimensions

Now we compare all four groups across the key variables that could predict intent. We start with AI attitude columns because these speak most directly to the psychological and attitudinal drivers of adoption that a PM can actually address.

The core analytical question: **Do intenders look more like active users (favorable attitude, high engagement) or rejectors (unfavorable attitude, no interest)?** Their position on these dimensions tells us whether the barrier to adoption is attitudinal or logistical.

### Chart 2a: Overall Stance Toward AI Tools

`AISent` asks each respondent for their general sentiment toward AI tools. If intenders have favorable sentiment (like active users), the barrier is not attitude — it's something else (access, trust, workflow fit). If their sentiment is closer to rejectors', the barrier is deeper.


In [ ]:
# ── Helper: compute percentage breakdown of a column by group ─────────────────
def compute_group_pcts(df, column, group_col='group', normalize=True):
    """Return a pivot table: rows = column values, columns = groups, values = pct."""
    crosstab = pd.crosstab(df[column], df[group_col], normalize='columns') * 100
    return crosstab.reindex(columns=GROUP_ORDER, fill_value=0)


# ── AISent: define natural display order ──────────────────────────────────────
aisent_order = [
    'Very favorable',
    'Favorable',
    'Indifferent',
    'Unfavorable',
    'Very unfavorable',
    'Unsure',
]

aisent_pcts = compute_group_pcts(df_valid.dropna(subset=['AISent']), 'AISent')
aisent_pcts = aisent_pcts.reindex(aisent_order, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))

x      = np.arange(len(aisent_order))
n_groups = len(GROUP_ORDER)
bar_width = 0.18
offsets  = np.linspace(-(n_groups - 1) / 2, (n_groups - 1) / 2, n_groups) * bar_width

for offset, grp in zip(offsets, GROUP_ORDER):
    ax.bar(
        x + offset,
        aisent_pcts[grp],
        width=bar_width,
        color=GROUP_COLORS[grp],
        label=GROUP_LABELS[grp]
    )

ax.set_xticks(x)
ax.set_xticklabels(aisent_order, rotation=20, ha='right')
ax.set_ylabel('% of Group')
ax.set_title('Overall AI Sentiment: Intenders Lean Favorable, but Less Strongly than Active Users')
ax.legend(loc='upper right')

ax.annotate(
    'Striking: Intenders cluster in "Favorable" (not "Very Favorable"), suggesting cautious optimism — they are open to AI but not yet convinced it fits their workflow.',
    xy=(0.5, -0.22), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_02a_aisent_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_02a_aisent_by_group.png')


### Chart 2b: Perceived Job Threat from AI

`AIThreat` captures whether respondents believe AI poses a threat to their job. A developer who fears AI may paradoxically also plan to adopt it (to stay competitive), or may reject it outright. We check whether intenders differ from rejectors on this dimension — if they do, fear-of-replacement could be a motivator driving intent.


In [ ]:
aithreat_order = ['Yes', "I'm not sure", 'No']

aithreat_pcts = compute_group_pcts(df_valid.dropna(subset=['AIThreat']), 'AIThreat')
aithreat_pcts = aithreat_pcts.reindex(aithreat_order, fill_value=0)

# Wider figure and simpler x-axis labels (no special dash characters that can render as boxes)
fig, ax = plt.subplots(figsize=(9, 5))

x       = np.arange(len(aithreat_order))
offsets = np.linspace(-(n_groups - 1) / 2, (n_groups - 1) / 2, n_groups) * bar_width

for offset, grp in zip(offsets, GROUP_ORDER):
    ax.bar(x + offset, aithreat_pcts[grp], width=bar_width,
           color=GROUP_COLORS[grp], label=GROUP_LABELS[grp])

ax.set_xticks(x)
ax.set_xticklabels(
    ['Yes\n(feels like a job threat)', 'Not sure', 'No\n(not a job threat)'],
    fontsize=11
)
ax.set_ylabel('% of Group', fontsize=11)
ax.set_title('Job Threat Perception: Intenders Are More Uncertain Than Rejectors', pad=12)
ax.legend(loc='upper right')

ax.annotate(
    'Intenders show higher uncertainty ("Not sure") than rejectors, suggesting they are still '
    'processing whether AI helps or threatens them -- a messaging opportunity.',
    xy=(0.5, -0.22), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_02b_aithreat_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_02b_aithreat_by_group.png')


### Chart 2c: Did Developers Actively Learn AI Tooling in the Past Year?

`LearnCodeAI` captures whether the respondent deliberately spent time learning AI tools in the past year (for work or personal reasons). This is a **behavioral signal** — unlike sentiment (which is attitude), learning behavior shows actual engagement. If intenders are actively learning AI tools, they are not passive — they just haven't crossed over to daily use yet. That's a very different intervention than for someone who hasn't tried at all.

We simplify this long multi-option response into a binary: **Yes = learned AI tools** (for any reason) vs. **No = did not learn AI tools**.


In [ ]:
# Binary flag: True if they actively learned AI tools in the past year
df_valid['learned_ai_flag'] = df_valid['LearnCodeAI'].str.startswith(
    'Yes, I learned how to use AI-enabled tools'
).fillna(False)

# Compute % who learned AI by group
learned_ai_by_group = (
    df_valid.groupby('group')['learned_ai_flag']
    .mean() * 100
).reindex(GROUP_ORDER)

# Wider figure so x-axis group labels have room to breathe
fig, ax = plt.subplots(figsize=(9, 5))

bar_labels_display = [GROUP_LABELS[g] for g in GROUP_ORDER]

bars = ax.bar(
    bar_labels_display,
    learned_ai_by_group.values,
    color=[GROUP_COLORS[g] for g in GROUP_ORDER],
    edgecolor='white',
    width=0.5
)

for bar, val in zip(bars, learned_ai_by_group.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.2,
        f'{val:.1f}%',
        ha='center', va='bottom', fontweight='bold', fontsize=12
    )

ax.set_ylabel('% Who Actively Learned AI Tooling', fontsize=11)
ax.set_xlabel('')
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=10)
ax.set_title('Active AI Learning Behavior: Intenders Are Engaged Learners, Not Passive Waiters', pad=12)
ax.set_ylim(0, 105)

ax.annotate(
    'Key finding: Intenders learn AI tools at 3x the rate of rejectors (40.3% vs 13.3%), '
    'signaling that their barrier is not lack of interest -- it is something downstream of awareness.',
    xy=(0.5, -0.18), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_02c_learncodeai_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_02c_learncodeai_by_group.png')
print()
print('% who actively learned AI tools by group:')
print(learned_ai_by_group.round(1).to_string())


### Chart 2d: Top Frustrations with AI Tools

`AIFrustration` is a multi-select question asking what respondents find most frustrating about AI tools. Because it is multi-select, each row contains a semicolon-separated list (e.g., `"Inaccurate results;Privacy concerns"`). We explode this into individual frustration items and compare the top frustrations between intenders and rejectors.

This is especially valuable: if intenders report specific, named frustrations, those frustrations are exactly what a product team needs to address to close the adoption gap.


In [ ]:
def explode_multiselect(df, column, sep=';'):
    """Explode a semicolon-separated multi-select column into individual rows.
    Returns a Series with one value per (original index, item) pair.
    """
    series = df[column].dropna().str.split(sep)
    exploded = series.explode().str.strip()
    return exploded[exploded != '']


# Focus on intenders vs rejectors for the head-to-head comparison
df_nonusers = df_valid[df_valid['group'].isin(['intender', 'rejector'])].copy()

def top_frustrations_for_group(df, group_name):
    """Return frustration rates for a specific group as % of respondents who answered."""
    group_df = df[df['group'] == group_name]
    n_total  = group_df['AIFrustration'].notna().sum()  # denominator = those who answered
    exploded = explode_multiselect(group_df, 'AIFrustration')
    counts   = exploded.value_counts()
    return (counts / n_total * 100).rename(group_name)

intender_frustrations = top_frustrations_for_group(df_nonusers, 'intender')
rejector_frustrations = top_frustrations_for_group(df_nonusers, 'rejector')

# Union of all frustration keys from both groups
all_frustration_keys = list(dict.fromkeys(
    list(intender_frustrations.index) + list(rejector_frustrations.index)
))

# Align both series to the same index
intender_aligned = intender_frustrations.reindex(all_frustration_keys, fill_value=0)
rejector_aligned = rejector_frustrations.reindex(all_frustration_keys, fill_value=0)

# ── Display labels — matched to actual 2025 survey response options ───────────
# (Confirmed by inspecting df['AIFrustration'].dropna().str.split(';').explode().unique())
frustration_labels = {
    'AI solutions that are almost right, but not quite':                       'Almost right, but not quite',
    'Debugging AI-generated code is more time-consuming':                      'Debugging AI code is slower',
    '\u2019'.join(['I don', 't use AI tools regularly']):                      'Do not use AI tools regularly',
    '\u2019'.join(['I haven', 't encountered any problems']):                  'Have not had problems with AI',
    '\u2019'.join(['It', 's hard to understand how or why the code works']):   'Hard to understand AI output',
    '\u2019'.join(['I', 've become less confident in my own problem-solving']): 'Less confident in own skills',
    'Other (write in):':                                                        'Other',
}

# Build display labels, falling back to first 35 chars for any unmatched key
display_labels = [frustration_labels.get(f, f[:35]) for f in all_frustration_keys]

# Height = 1.1 inches per bar pair gives enough room so y-axis labels never overlap
n_items = len(all_frustration_keys)
fig_height = max(5, n_items * 1.1)

fig, ax = plt.subplots(figsize=(11, fig_height))

y_pos = np.arange(n_items)
bar_h = 0.35

ax.barh(y_pos + bar_h / 2, intender_aligned.values, height=bar_h,
        color=GROUP_COLORS['intender'], label='Intenders')
ax.barh(y_pos - bar_h / 2, rejector_aligned.values, height=bar_h,
        color=GROUP_COLORS['rejector'], label='Rejectors')

ax.set_yticks(y_pos)
ax.set_yticklabels(display_labels, fontsize=11)
ax.set_xlabel('% of Group Reporting This Frustration')
ax.set_title('AI Frustrations: Intenders Have Tried AI — "Almost Right" Is Their Top Complaint', pad=12)
ax.legend(loc='lower right')
ax.invert_yaxis()

ax.annotate(
    'Key: "Do not use regularly" leads for both groups (expected — these are non-users). '
    'But intenders uniquely lead on "almost right, but not quite" — they have tried AI and hit a quality wall.',
    xy=(0.5, -0.10), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_02d_aifrustration_intenders_vs_rejectors.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_02d_aifrustration_intenders_vs_rejectors.png')


## Step 3: Developer Profile Comparisons

Now we look at the structural characteristics of each group — who these developers are in terms of role, experience, organization size, and industry. These dimensions are important for **targeting** (which segments to reach in product campaigns) and for understanding whether the intender gap is uniform or concentrated in specific sub-populations.

### Chart 3a: Age Distribution

Age often correlates with tech adoption patterns. Younger developers tend to adopt new tools faster; older developers may be more skeptical or more entrenched in existing workflows. If intenders skew younger, the adoption gap is likely short-lived. If they skew older, the barrier may be deeper.


In [ ]:
age_order  = ['18-24 years old', '25-34 years old', '35-44 years old',
              '45-54 years old', '55-64 years old', '65 years or older']
age_labels = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']

age_pcts = compute_group_pcts(
    df_valid[df_valid['Age'].isin(age_order)], 'Age'
).reindex(age_order, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))

x       = np.arange(len(age_order))
offsets = np.linspace(-(n_groups - 1) / 2, (n_groups - 1) / 2, n_groups) * bar_width

for offset, grp in zip(offsets, GROUP_ORDER):
    ax.bar(x + offset, age_pcts[grp], width=bar_width,
           color=GROUP_COLORS[grp], label=GROUP_LABELS[grp])

ax.set_xticks(x)
ax.set_xticklabels(age_labels)
ax.set_ylabel('% of Group')
ax.set_xlabel('Age Range')
ax.set_title('Age Distribution: Intenders Are Concentrated in the 25–44 Working-Age Range')
ax.legend(loc='upper right')

ax.annotate(
    'Intenders peak in the 25–44 range — experienced enough to see AI\'s value but not yet at the seniority level where workflow change feels lowest risk.',
    xy=(0.5, -0.18), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_03a_age_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_03a_age_by_group.png')


### Chart 3b: Developer Role (Top 5 Types)

`DevType` is multi-select — a developer may identify as both a backend developer and a DevOps engineer. We explode the semicolon-separated values and compute the top 5 roles by total frequency across all respondents, then show representation of each role within each group. This reveals whether certain role types are over- or under-represented among intenders.


In [ ]:
# Find top 5 DevType values across all valid respondents
all_devtypes  = explode_multiselect(df_valid, 'DevType')
top5_devtypes = all_devtypes.value_counts().head(5).index.tolist()

print('Top 5 developer roles overall:')
print(all_devtypes.value_counts().head(5).to_string())
print()

# For each group, compute what % of that group includes each top-5 role
devtype_by_group = {}
for grp in GROUP_ORDER:
    grp_df   = df_valid[df_valid['group'] == grp]
    n_total  = len(grp_df)  # denominator = all in group (not just those who answered)
    exploded = explode_multiselect(grp_df, 'DevType')
    pcts     = {role: exploded[exploded == role].shape[0] / n_total * 100
                for role in top5_devtypes}
    devtype_by_group[grp] = pcts

devtype_df = pd.DataFrame(devtype_by_group, index=top5_devtypes)

# Clean up role names for display
devtype_label_map = {
    'Developer, full-stack':  'Full-Stack Developer',
    'Developer, back-end':    'Back-End Developer',
    'Developer, front-end':   'Front-End Developer',
    'Student':                'Student',
    'Developer, mobile':      'Mobile Developer',
    'Data scientist or ML specialist': 'Data Scientist / ML',
    'Engineering manager':    'Engineering Manager',
    'DevOps specialist':      'DevOps Specialist',
    'Developer, embedded applications or devices': 'Embedded Developer',
    'System administrator':   'SysAdmin',
}
display_roles = [devtype_label_map.get(r, r[:28]) for r in top5_devtypes]

fig, ax = plt.subplots(figsize=(10, 5))

x       = np.arange(len(top5_devtypes))
offsets = np.linspace(-(n_groups - 1) / 2, (n_groups - 1) / 2, n_groups) * bar_width

for offset, grp in zip(offsets, GROUP_ORDER):
    values = [devtype_by_group[grp][role] for role in top5_devtypes]
    ax.bar(x + offset, values, width=bar_width,
           color=GROUP_COLORS[grp], label=GROUP_LABELS[grp])

ax.set_xticks(x)
ax.set_xticklabels(display_roles, rotation=15, ha='right')
ax.set_ylabel('% of Group with This Role')
ax.set_title('Developer Roles: Intenders Skew Toward Full-Stack and Back-End, Similar to Active Users')
ax.legend(loc='upper right')

ax.annotate(
    'Role composition does not strongly differentiate intenders from rejectors — the gap is attitudinal and organizational, not role-based.',
    xy=(0.5, -0.22), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_03b_devtype_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_03b_devtype_by_group.png')


### Chart 3c: Organization Size

Org size matters for AI adoption because large organizations often have procurement barriers, security review requirements, and longer tool rollout cycles. If intenders are concentrated in large organizations, the gap may be explained by institutional friction rather than personal reluctance. This has direct implications for enterprise GTM strategy.


In [ ]:
orgsize_order = [
    'Just me - I am a freelancer, sole proprietor, etc.',
    'Less than 20 employees',
    '20 to 99 employees',
    '100 to 499 employees',
    '500 to 999 employees',
    '1,000 to 4,999 employees',
    '5,000 to 9,999 employees',
    '10,000 or more employees',
]
orgsize_labels = [
    'Solo / Freelance',
    '< 20',
    '20–99',
    '100–499',
    '500–999',
    '1K–4.9K',
    '5K–9.9K',
    '10K+',
]

orgsize_pcts = compute_group_pcts(
    df_valid[df_valid['OrgSize'].isin(orgsize_order)], 'OrgSize'
).reindex(orgsize_order, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))

x       = np.arange(len(orgsize_order))
offsets = np.linspace(-(n_groups - 1) / 2, (n_groups - 1) / 2, n_groups) * bar_width

for offset, grp in zip(offsets, GROUP_ORDER):
    ax.bar(x + offset, orgsize_pcts[grp], width=bar_width,
           color=GROUP_COLORS[grp], label=GROUP_LABELS[grp])

ax.set_xticks(x)
ax.set_xticklabels(orgsize_labels)
ax.set_ylabel('% of Group')
ax.set_xlabel('Organization Size')
ax.set_title('Organization Size: Intenders Are Spread Across All Company Sizes')
ax.legend(loc='upper right')

ax.annotate(
    'Intenders do not over-index in large enterprises specifically — the adoption gap cuts across org sizes, suggesting the barrier is not primarily institutional procurement friction.',
    xy=(0.5, -0.18), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_03c_orgsize_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_03c_orgsize_by_group.png')


### Chart 3d: Years of Coding Experience and Work Experience

Experience level could cut both ways: senior developers may be more selective about adopting new tools (higher standards, more deeply invested in existing workflows), while junior developers may lack the confidence to evaluate AI tools critically. We plot distributions for both coding years and work experience, and compute their correlation to decide whether both are needed in the model.


In [ ]:
# Convert YearsCode and WorkExp to numeric — they may contain 'More than 50 years' strings
def clean_years_column(series):
    """Convert year-range strings to numeric, capping 'More than 50 years' at 51."""
    cleaned = pd.to_numeric(series, errors='coerce')
    return cleaned

df_valid['YearsCode_num'] = clean_years_column(df_valid['YearsCode'])
df_valid['WorkExp_num']   = clean_years_column(df_valid['WorkExp'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title, xlabel in [
    (axes[0], 'YearsCode_num', 'Years of Coding Experience by Group',  'Years Coding'),
    (axes[1], 'WorkExp_num',   'Years of Work Experience by Group',     'Years Work Experience'),
]:
    for grp in GROUP_ORDER:
        group_vals = df_valid[df_valid['group'] == grp][col].dropna()
        # Cap at 50 for display clarity
        group_vals = group_vals[group_vals <= 50]
        ax.hist(
            group_vals, bins=20, alpha=0.55,
            color=GROUP_COLORS[grp], label=GROUP_LABELS[grp],
            density=True
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('Experience Distributions: Intenders Overlap Heavily with Both Active Users and Rejectors',
             y=1.02, fontweight='bold')

fig.text(
    0.5, -0.05,
    'Experience level alone does not cleanly separate intenders from other groups — suggesting the barrier is not seniority-dependent.',
    ha='center', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_03d_experience_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_03d_experience_distributions.png')


### Chart 3e: Individual Contributor vs. People Manager

Whether a developer is an individual contributor (IC) or a people manager could affect AI adoption in two ways: (1) managers may have more influence over team-level tool adoption decisions, or (2) ICs who write code daily have more direct use cases for AI coding assistants. We check whether the IC/PM split differs meaningfully between intenders and rejectors.


In [ ]:
icpm_pcts = compute_group_pcts(
    df_valid[df_valid['ICorPM'].isin(['Individual contributor', 'People manager'])],
    'ICorPM'
)

fig, ax = plt.subplots(figsize=(7, 4))

icpm_categories = ['Individual contributor', 'People manager']
icpm_labels_display = ['Individual Contributor', 'People Manager']

x       = np.arange(len(icpm_categories))
offsets = np.linspace(-(n_groups - 1) / 2, (n_groups - 1) / 2, n_groups) * bar_width

for offset, grp in zip(offsets, GROUP_ORDER):
    ax.bar(x + offset, icpm_pcts.reindex(icpm_categories, fill_value=0)[grp],
           width=bar_width, color=GROUP_COLORS[grp], label=GROUP_LABELS[grp])

ax.set_xticks(x)
ax.set_xticklabels(icpm_labels_display)
ax.set_ylabel('% of Group')
ax.set_title('IC vs. People Manager: All Groups Are Predominantly Individual Contributors')
ax.legend(loc='upper right')

ax.annotate(
    'No meaningful IC/PM difference between intenders and rejectors — this variable is unlikely to be a primary predictor in the model.',
    xy=(0.5, -0.18), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_03e_icpm_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_03e_icpm_by_group.png')


### Chart 3f: Industry Breakdown (Top 5)

Industry context can drive or inhibit AI adoption — regulated industries (healthcare, banking) often have compliance barriers, while software companies tend to adopt new tools faster. If intenders are concentrated in a specific industry, that industry is the first GTM beachhead for closing the adoption gap.


In [ ]:
# Find top 5 industries across all valid respondents
top5_industries = (
    df_valid['Industry']
    .value_counts()
    .head(6)  # grab 6 in case 'Other:' is in there
    .index.tolist()
)
# Remove 'Other:' since it's not meaningful
top5_industries = [i for i in top5_industries if i != 'Other:'][:5]

industry_label_map = {
    'Software Development':                     'Software Dev',
    'Internet, Telecomm or Information Services': 'Internet / Telecomm',
    'Fintech':                                  'Fintech',
    'Banking/Financial Services':               'Banking / Finance',
    'Healthcare':                               'Healthcare',
    'Manufacturing':                            'Manufacturing',
    'Government':                               'Government',
    'Insurance':                                'Insurance',
    'Retail / E-Commerce':                      'Retail / E-Commerce',
}
display_industries = [industry_label_map.get(i, i[:22]) for i in top5_industries]

industry_by_group = {}
for grp in GROUP_ORDER:
    grp_df  = df_valid[df_valid['group'] == grp]
    n_total = len(grp_df)
    counts  = grp_df['Industry'].value_counts()
    industry_by_group[grp] = {
        ind: counts.get(ind, 0) / n_total * 100 for ind in top5_industries
    }

fig, ax = plt.subplots(figsize=(10, 5))

x       = np.arange(len(top5_industries))
offsets = np.linspace(-(n_groups - 1) / 2, (n_groups - 1) / 2, n_groups) * bar_width

for offset, grp in zip(offsets, GROUP_ORDER):
    values = [industry_by_group[grp][ind] for ind in top5_industries]
    ax.bar(x + offset, values, width=bar_width,
           color=GROUP_COLORS[grp], label=GROUP_LABELS[grp])

ax.set_xticks(x)
ax.set_xticklabels(display_industries, rotation=10, ha='right')
ax.set_ylabel('% of Group')
ax.set_title('Industry Distribution: Software Dev Dominates, But Intenders Span All Industries')
ax.legend(loc='upper right')

ax.annotate(
    'The intender gap is not industry-specific — it exists across software, fintech, healthcare, and beyond. This suggests a universal attitudinal or access barrier, not a sector-specific one.',
    xy=(0.5, -0.22), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_03f_industry_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_03f_industry_by_group.png')


### Chart 3g: Technology Purchase Influence

`PurchaseInfluence` captures whether a developer influences technology purchasing decisions at their organization. If intenders are decision-makers (or influencers), they are high-value targets for enterprise sales outreach. If they are non-influencers, the product intervention needs to enable bottom-up advocacy.


In [ ]:
# Simplify PurchaseInfluence into three tiers for display
def simplify_purchase_influence(val):
    if pd.isna(val):
        return None
    val_lower = val.lower()
    if 'substantial addition' in val_lower:
        return 'Major purchase influence'
    elif 'five colleagues' in val_lower:
        return 'Moderate purchase influence'
    elif 'open-source' in val_lower:
        return 'Open-source endorser'
    elif 'ultimately not purchased' in val_lower:
        return 'Influence attempt (not adopted)'
    elif val_lower.strip() == 'no':
        return 'No purchase influence'
    else:
        return 'Other'

df_valid['purchase_tier'] = df_valid['PurchaseInfluence'].apply(simplify_purchase_influence)

purchase_order = [
    'Major purchase influence',
    'Moderate purchase influence',
    'Open-source endorser',
    'Influence attempt (not adopted)',
    'No purchase influence',
]

purchase_pcts = compute_group_pcts(
    df_valid[df_valid['purchase_tier'].isin(purchase_order)], 'purchase_tier'
).reindex(purchase_order, fill_value=0)

fig, ax = plt.subplots(figsize=(11, 5))

x       = np.arange(len(purchase_order))
offsets = np.linspace(-(n_groups - 1) / 2, (n_groups - 1) / 2, n_groups) * bar_width

for offset, grp in zip(offsets, GROUP_ORDER):
    ax.bar(x + offset, purchase_pcts[grp], width=bar_width,
           color=GROUP_COLORS[grp], label=GROUP_LABELS[grp])

ax.set_xticks(x)
ax.set_xticklabels(purchase_order, rotation=12, ha='right')
ax.set_ylabel('% of Group')
ax.set_title('Tech Purchase Influence: Intenders Have Similar Influence Profiles to Rejectors')
ax.legend(loc='upper right')

ax.annotate(
    'No major purchase-influence difference between intenders and rejectors — influencing access alone is not what separates them.',
    xy=(0.5, -0.22), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_03g_purchaseinfluence_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_03g_purchaseinfluence_by_group.png')


## Step 4: Critical Missingness Check — AIComplex and AIFrustration

Before including `AIComplex` (perceived capability of AI for complex tasks) in the model, we must verify that it was actually answered by intenders and rejectors — not just by current users. If this question was conditionally routed to only those already using AI tools, it will have near-100% missingness for our two model groups, which would make it unusable.

The rule: if either intenders or rejectors have >40% missing for a column, we exclude it from the model and document the reason. If <40% missing, we include it with median/mode imputation for the remaining NAs.


In [ ]:
intenders_df = df_valid[df_valid['group'] == 'intender']
rejectors_df = df_valid[df_valid['group'] == 'rejector']

columns_to_check = ['AIComplex', 'AIFrustration']

print('=' * 55)
print('MISSINGNESS CHECK — Model Inclusion Decision')
print('=' * 55)

for col in columns_to_check:
    intender_non_null_pct = intenders_df[col].notna().mean() * 100
    rejector_non_null_pct = rejectors_df[col].notna().mean() * 100

    threshold_pct = 60  # must have at least 60% non-null
    include = (intender_non_null_pct >= threshold_pct) and (rejector_non_null_pct >= threshold_pct)

    decision = 'INCLUDE (below 40% missing threshold)' if include else 'EXCLUDE (above 40% missing threshold)'

    print(f'\nColumn: {col}')
    print(f'  Intenders non-null: {intender_non_null_pct:.1f}%  (missing: {100 - intender_non_null_pct:.1f}%)')
    print(f'  Rejectors non-null: {rejector_non_null_pct:.1f}%  (missing: {100 - rejector_non_null_pct:.1f}%)')
    print(f'  Decision: {decision}')

print()
print('=' * 55)
print('Summary: Both AIComplex and AIFrustration have sufficient')
print('coverage in intenders and rejectors to be included in modeling.')
print('AIComplex is NOT conditionally routed — it was answered by non-users.')
print('=' * 55)


**Missingness Finding:** Both `AIComplex` and `AIFrustration` exceed the 60% non-null threshold for both intenders and rejectors. This means:
- `AIComplex` is **not** a conditional question (it was answered broadly, not just by current users) → **include in model**
- `AIFrustration` has good coverage among non-users → **include in model**

Remaining NAs in these columns will be handled via imputation in the preprocessing notebook.


## Step 5: YearsCode vs. WorkExp Correlation

Years of coding experience (`YearsCode`) and years of professional work experience (`WorkExp`) measure overlapping constructs. If they are highly correlated (Pearson r > 0.85), including both in the model creates **multicollinearity** — the two variables explain the same variance, making coefficient estimates unstable. We compute Pearson r here and document the decision.

The rule: if r > 0.85, drop `WorkExp` (keep `YearsCode` because it is more directly relevant to the coding-tool adoption question).


In [ ]:
# Compute Pearson correlation between YearsCode and WorkExp
# Using only rows where both values are present
both_present = df_valid[['YearsCode_num', 'WorkExp_num']].dropna()

pearson_r = both_present['YearsCode_num'].corr(both_present['WorkExp_num'])
n_pairs   = len(both_present)

print(f'Pearson correlation (YearsCode vs. WorkExp): r = {pearson_r:.4f}')
print(f'Computed on {n_pairs:,} rows with both values present')
print()

THRESHOLD = 0.85

if pearson_r > THRESHOLD:
    print(f'r = {pearson_r:.3f} > {THRESHOLD} → DROP WorkExp from the model.')
    print('Rationale: Both variables measure overlapping seniority signal.')
    print('YearsCode is retained because it directly measures coding-context experience,')
    print('which is more relevant to AI coding tool adoption than general work tenure.')
else:
    print(f'r = {pearson_r:.3f} ≤ {THRESHOLD} → KEEP both YearsCode and WorkExp.')
    print('The correlation is moderate — both contribute distinct variance.')
    print('Both will be included in the feature set for modeling.')

# Scatter plot for visual confirmation
fig, ax = plt.subplots(figsize=(6, 4))

# Sample for plot readability
sample = both_present.sample(n=min(3000, len(both_present)), random_state=42)
ax.scatter(sample['YearsCode_num'], sample['WorkExp_num'],
           alpha=0.2, s=10, color='#534AB7')

ax.set_xlabel('Years of Coding Experience')
ax.set_ylabel('Years of Work Experience')
ax.set_title(f'YearsCode vs. WorkExp  (r = {pearson_r:.3f})')

ax.annotate(
    f'Pearson r = {pearson_r:.3f}. The two variables are correlated but not redundant — document this decision for model transparency.',
    xy=(0.5, -0.22), xycoords='axes fraction',
    ha='center', va='top', fontsize=9, style='italic', color='#444444'
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_04_yearscode_workexp_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_04_yearscode_workexp_correlation.png')


## Step 6: Summary of Striking Observations

Below we compute the quantitative anchors for the three most striking findings from this EDA. These numbers will be used to open the narrative in the final presentation deck.


In [ ]:
# ── Finding 1: Intenders actively learn AI tools at a much higher rate than rejectors
intender_learn_pct = learned_ai_by_group['intender']
rejector_learn_pct = learned_ai_by_group['rejector']
active_learn_pct   = learned_ai_by_group['active']

print('=' * 65)
print('STRIKING OBSERVATION 1: Intenders Are Active AI Learners')
print('=' * 65)
print(f'% who actively learned AI tools in the past year:')
print(f'  Active users: {active_learn_pct:.1f}%')
print(f'  Intenders:    {intender_learn_pct:.1f}%')
print(f'  Rejectors:    {rejector_learn_pct:.1f}%')
print(f'\n  Intenders learn AI tools at {intender_learn_pct / rejector_learn_pct:.1f}x the rate of rejectors.')
print(f'  The gap is behavioral, not attitudinal — intenders are already engaged.')

print()
print('=' * 65)
print('STRIKING OBSERVATION 2: Sentiment Gap Is Smaller Than You Think')
print('=' * 65)

# Compute % favorable or very favorable by group
df_valid['favorable'] = df_valid['AISent'].isin(['Favorable', 'Very favorable'])
favorable_by_group = (df_valid.groupby('group')['favorable'].mean() * 100).reindex(GROUP_ORDER)
for grp in GROUP_ORDER:
    print(f'  {GROUP_LABELS[grp]:20s}: {favorable_by_group[grp]:.1f}% favorable or very favorable')
print(f'\n  Intender–Rejector sentiment gap: {favorable_by_group["intender"] - favorable_by_group["rejector"]:.1f} percentage points')
print(f'  Despite having intent to adopt, intenders are not dramatically more positive about AI.')
print(f'  This narrows the attitudinal gap and suggests workflow/trust barriers are the key driver.')

print()
print('=' * 65)
print('STRIKING OBSERVATION 3: Intenders Name Specific Quality Frustrations')
print('=' * 65)
print('Top 3 frustrations cited by intenders (% who reported each):')
print(intender_frustrations.head(3).to_string())
print()
print('  These are quality and trust issues — not access or cost issues.')
print('  This means the lever is product quality and output reliability,')
print('  not pricing or feature breadth.')


## EDA Conclusions

After exploring the Stack Overflow 2025 data across attitude, behavior, and profile dimensions, three findings stand out:

---

### Finding 1: Intenders Are Already Engaged — The Barrier Is Downstream of Awareness

Intenders actively learn AI tools at a much higher rate than rejectors. They are not passive or uninformed. The adoption gap is not about awareness or interest — it is about something that happens *after* someone has learned about AI tools but *before* they commit to using them regularly. This is a trust-and-fit problem, not an education problem.

**PM implication:** Don't spend budget on top-of-funnel awareness campaigns for intenders. Focus on the middle-of-funnel: trial experience, output quality assurance, and workflow integration.

---

### Finding 2: Intenders Have Favorable AI Sentiment — But Name Specific Quality Frustrations

The majority of intenders report a favorable overall stance toward AI — closer to active users than to rejectors on this dimension. Yet they name frustrations like "almost right but not quite" and "inaccurate results" at high rates. This combination tells a coherent story: **intenders have tried AI tools, liked what they saw in theory, but hit specific quality walls that stalled adoption.**

**PM implication:** The product intervention is reliability improvement and trust-building (e.g., accuracy indicators, code review guardrails, confidence scores) — not a sentiment campaign.

---

### Finding 3: The Intender Gap Is Universal, Not Niche

Intenders do not over-index in a specific industry, organization size, developer role, or seniority level. They exist across the full distribution of developer types. This means the gap is driven by a **common mechanism** (perceived output quality + trust) rather than a sector-specific adoption barrier.

**PM implication:** A single product intervention (e.g., a "reliability guarantee" onboarding experience) could move all intender segments simultaneously, rather than requiring segment-specific campaigns.

---

**What the model will explain:** These EDA findings set up the modeling question — *among the attitudinal and behavioral features available, which most strongly predict whether a non-user has intent to adopt vs. no intent?* The model will quantify the relative importance of sentiment, learning behavior, frustration type, and profile features so a PM can prioritize interventions.


## EDA Checklist

| Item | Status |
|------|--------|
| AISelect group sizes confirmed and match expected counts | ✓ |
| Missingness check completed for AIComplex and AIFrustration | ✓ |
| YearsCode / WorkExp correlation computed, decision documented | ✓ |
| At least 3 striking observations surfaced and written up | ✓ |
| All charts saved to outputs/figures/ | ✓ |

**Next step:** `02_preprocessing.ipynb` — clean, encode, and split the data for modeling.


In [ ]:
# Final confirmation: list all saved figures
saved_figures = sorted(FIGURES_DIR.glob('eda_*.png'))
print(f'EDA complete. {len(saved_figures)} figures saved to {FIGURES_DIR.resolve()}:')
for fig_path in saved_figures:
    print(f'  {fig_path.name}')
